# Amazon ML Challenge: Business Entity Resolution - Kaggle runner

End-to-end runner for the submission pipeline on Kaggle (CPU or GPU kernel, internet enabled).

What this notebook does, in order:

1. Installs the few extra dependencies (`gdown`, `rapidfuzz`, `lightgbm`).
2. Locates the dataset: uses a Kaggle input dataset if attached, otherwise downloads the zip from Google Drive and unzips it.
3. Fetches `er_pipeline.py` plus the validator / scorer from the GitHub repo.
4. Runs the pipeline: normalisation -> TF-IDF char n-gram blocking -> pair features -> LightGBM matcher -> F0.5-tuned threshold -> `matching_results.tsv` + `candidate_pairs.tsv`.
5. Validates the outputs against the official format rules.
6. Zips the full submission package (`output/`, `code/business_entity_resolution/`, `Documentation_template.md`).

No external data lookup is performed; only the provided train/test TSVs are used.


In [ ]:
!pip install -q gdown rapidfuzz lightgbm


In [ ]:
import os
import subprocess
from glob import glob

GDRIVE_ID = "1bukugde70Drs9bHw8nr5oSr72ZPHDHzc"
ZIP_PATH = "/kaggle/working/data.zip"
DATA_ROOT = "/kaggle/working/data"


def find_train_source1(root):
    hits = glob(os.path.join(root, "**", "train_source1.tsv"), recursive=True)
    return sorted(hits, key=len)[0] if hits else None


# 1) Prefer a dataset attached as a Kaggle input.
hit = find_train_source1("/kaggle/input") if os.path.isdir("/kaggle/input") else None

# 2) Otherwise download the zip from Google Drive (skip if already present) and unzip.
if hit is None:
    if not os.path.exists(ZIP_PATH):
        import gdown
        gdown.download(id=GDRIVE_ID, output=ZIP_PATH, quiet=False)
    else:
        print("zip already present:", ZIP_PATH)
    os.makedirs(DATA_ROOT, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", DATA_ROOT], check=True)
    print(subprocess.run(["find", DATA_ROOT, "-maxdepth", "4"], capture_output=True, text=True).stdout)
    hit = find_train_source1(DATA_ROOT)

assert hit is not None, "train_source1.tsv not found under /kaggle/input or the unzipped download"

# DATA_DIR is the directory containing train/ and test/ (parent of parent of train_source1.tsv).
DATA_DIR = os.path.dirname(os.path.dirname(hit))
TEST_DIR = os.path.join(DATA_DIR, "test")
print("DATA_DIR =", DATA_DIR)
print("TEST_DIR =", TEST_DIR)
print(sorted(os.listdir(DATA_DIR)))


In [ ]:
import os
import urllib.request

RAW_BASE = "https://raw.githubusercontent.com/Manureddy148/Ai/claude/wizardly-cori-snj0nx"
WORK = "/kaggle/working"

FILES = {
    "code/business_entity_resolution/src/er_pipeline.py": f"{WORK}/er_pipeline.py",
    "utils/validate_submission.py": f"{WORK}/validate_submission.py",
    "utils/score.py": f"{WORK}/score.py",
}


def fetch(rel_path, dest):
    url = f"{RAW_BASE}/{rel_path}"
    try:
        urllib.request.urlretrieve(url, dest)
        print(f"fetched {url} -> {dest} ({os.path.getsize(dest)} bytes)")
        return True
    except Exception as e:  # noqa: BLE001
        print(f"FAILED to fetch {url}: {e}")
        return False


missing = [dest for rel, dest in FILES.items() if not fetch(rel, dest)]

if missing:
    print("\nFallback: internet may be disabled for this kernel. Upload these files manually\n"
          "(Kaggle sidebar -> Add data -> Upload, or drag into /kaggle/working) and re-run:")
    for m in missing:
        print("  ", m)

assert os.path.exists(f"{WORK}/er_pipeline.py"), "er_pipeline.py is required to continue"


In [ ]:
OUT_DIR = "/kaggle/working/output"
!python /kaggle/working/er_pipeline.py --data-dir {DATA_DIR} --out-dir {OUT_DIR}
!ls -la {OUT_DIR}


In [ ]:
!python /kaggle/working/validate_submission.py \
    --matching {OUT_DIR}/matching_results.tsv \
    --candidate {OUT_DIR}/candidate_pairs.tsv \
    --test-dir {TEST_DIR}

# Optional: score.py computes the official macro F0.5 for any prediction file in
# train-ground-truth format (the pipeline itself only prints its OOF score to the log;
# it does not write such a file). Uncomment if you produce one:
# !python /kaggle/working/score.py --pred {OUT_DIR}/train_oof_predictions.tsv \
#     --truth {DATA_DIR}/train/train_ground_truth.tsv --source1 {DATA_DIR}/train/train_source1.tsv --by-size


In [ ]:
import os
import shutil
import urllib.request

TEAM = "team"  # <-- change to your team name
PKG = f"/kaggle/working/{TEAM}_submission"
CODE_DIR = f"{PKG}/code/business_entity_resolution"

shutil.rmtree(PKG, ignore_errors=True)
os.makedirs(f"{PKG}/output", exist_ok=True)
os.makedirs(f"{CODE_DIR}/src", exist_ok=True)

# outputs
for name in ("matching_results.tsv", "candidate_pairs.tsv"):
    shutil.copy(f"{OUT_DIR}/{name}", f"{PKG}/output/{name}")

# code
shutil.copy("/kaggle/working/er_pipeline.py", f"{CODE_DIR}/src/er_pipeline.py")

# README / requirements / documentation from the same GitHub base
for rel, dest in {
    "code/business_entity_resolution/README.md": f"{CODE_DIR}/README.md",
    "code/business_entity_resolution/requirements.txt": f"{CODE_DIR}/requirements.txt",
    "Documentation_template.md": f"{PKG}/Documentation_template.md",
}.items():
    try:
        urllib.request.urlretrieve(f"{RAW_BASE}/{rel}", dest)
        print("fetched", rel)
    except Exception as e:  # noqa: BLE001
        print(f"WARNING: could not fetch {rel} ({e}); add it to the package manually")

ZIP_OUT = shutil.make_archive(PKG, "zip", root_dir=PKG)
print("created", ZIP_OUT, os.path.getsize(ZIP_OUT), "bytes")
!unzip -l {ZIP_OUT}


## Downloading the results

After the run finishes, everything lives under `/kaggle/working`:

- `/kaggle/working/output/matching_results.tsv` and `candidate_pairs.tsv` - the two prediction files.
- `/kaggle/working/<team>_submission.zip` - the complete package (`output/`, `code/business_entity_resolution/{src/er_pipeline.py, README.md, requirements.txt}`, `Documentation_template.md`).

To download: open the **Output** tab in the right-hand panel of the Kaggle editor (or click **Save Version** and then use the Output section of the saved run) and download the zip. Files under `/kaggle/working` are also browsable from the *Data -> Output* sidebar.

What to upload to the portal:

1. `matching_results.tsv` and `candidate_pairs.tsv` where the portal asks for the prediction files.
2. `<team>_submission.zip` as the final code + documentation package.

Before uploading, fill in the team-specific sections of `Documentation_template.md` and make sure the validator cell above reported no errors.
